In [ ]:
%%capture
!pip install -r requirements.txt
from utils import *

wd = WarpDrive()

dataset_type = wd.get_args("dataset_type")

In [ ]:
## Helper functions
def read_parquet_from_azure(container, directory):
    with wd.store.get_object_readable_stream(container, directory) as stream:
        data = pd.read_parquet(stream)
    return data

def extract_windows(pargs):
    window_definitions = []
    for window in pargs.get("windows", {}).get("definition", []):
        for window_name, window_values in window.items():
            window_definitions.append({
                "window":     window_name,
                "start_date": window_values.get("start_date"),
                "end_date":   window_values.get("end_date")
            })
    return window_definitions

def add_segment_feature(df_features, new_feature_df, key='customer_id'):
    return df_features.merge(new_feature_df, on=key, how='left')

def safe_div(num, den):
    den_safe = den.mask(den == 0, pd.NA)
    return num / den_safe

def save_df_to_azure(data, container_name, file_path, output_type='parquet'):
    buffer = io.BytesIO()
    if output_type == 'parquet':
        data.to_parquet(buffer, index=False)
    buffer.seek(0)
    wd.store.put_object(container_name=container_name, key=file_path, data=buffer)
    print(f"Saved → {container_name}/{file_path}")


In [ ]:
import json
pargs = json.loads(wd.store.read_object(
    'nimbus-uno-usbank',
    'anomaly_detection/retail_banking/arguments/pipeline_arguments.json'
).decode('utf-8'))

container        = pargs['paths']['container']
prefix_base      = pargs['paths']['prefix_base']
total_num_windows = pargs['windows']['count']
start_date       = pargs['data_period']['start_date']
end_date         = pargs['data_period']['end_date']
windows_info     = extract_windows(pargs)

file_path        = 'anomaly_detection/retail_banking_new/data'
date_folder_fmt  = '_'.join([start_date.replace('-', ''), end_date.replace('-', '')])
DATA_PATH       = f"{file_path}/{date_folder_fmt}"

In [ ]:
CASH_CHANNELS    = ['Branch', 'Agent']
ATM_CHANNELS     = ['ATM']
DIGITAL_CHANNELS = ['Online Banking', 'Mobile App']
ACH_CHANNELS     = ['ACH']
WIRE_CHANNELS    = ['Wire']
CARD_CHANNELS    = ['POS']

CHANNEL_MAP = {
    'cash':    CASH_CHANNELS,
    'atm':     ATM_CHANNELS,
    'digital': DIGITAL_CHANNELS,
    'ach':     ACH_CHANNELS,
    'wire':    WIRE_CHANNELS,
    'card':    CARD_CHANNELS,
}

In [ ]:
# HOLIDAYS = pd.to_datetime([
#     '2024-01-01', '2024-07-04', '2024-11-28', '2024-12-25',
#     '2025-01-01', '2025-07-04', '2025-11-27', '2025-12-25',
#     '2026-01-01',
# ])

In [ ]:
def tag_channel(df: pd.DataFrame) -> pd.DataFrame:
    """
    Adds 'channel_group' column (cash / atm / digital / ach / wire / card / other)
    from transaction_channel.  Applied once per window so feature functions can
    filter by a single string.
    """
    df = df.copy()
    df['channel_group'] = 'other'
    for label, vals in CHANNEL_MAP.items():
        df.loc[df['transaction_channel'].isin(vals), 'channel_group'] = label
    return df
 
 
def prep_window_df(df_raw: pd.DataFrame) -> pd.DataFrame:
    """
    Standard pre-processing applied to every raw window file:
      • Parse transaction_datetime
      • Derive txn_month (Period[M])
      • Add is_round_amount flag (amount divisible by 100)
      • Tag channel_group
    Returns a clean copy ready for feature building.
    """
    df = df_raw.copy()
    df['transaction_datetime'] = pd.to_datetime(df['transaction_datetime'], errors='coerce')
    df['txn_month']            = df['transaction_datetime'].dt.to_period('M')
    df['is_round_amount']      = (df['transaction_amount_usd'] % 100 == 0).astype(int)
    df = tag_channel(df)
    return df

In [ ]:
## Percentage change

def build_pct_change(df: pd.DataFrame, window_end: pd.Timestamp) -> pd.DataFrame:
    """
    For each (channel_group × debit_credit_indicator) pair:
        pct_change = (latest_month_amount − mean_prior_months_amount)
                     / (mean_prior_months_amount + ε)
    Same formula applied to is_international transactions as a pseudo-channel.
 
    Feature naming: {channel}_{direction}_pct_change
                    international_{direction}_pct_change
    """
    monthly = (
        df.groupby(
            ['customer_id', 'txn_month', 'channel_group', 'debit_credit_indicator'],
            as_index=False
        )['transaction_amount_usd'].sum()
    )
    latest_month = monthly['txn_month'].max()
 
    def _pct(sub: pd.DataFrame) -> pd.DataFrame:
        latest   = sub[sub['txn_month'] == latest_month]
        hist     = sub[sub['txn_month'] <  latest_month]
        hist_base = (
            hist
            .groupby(['customer_id', 'channel_group', 'debit_credit_indicator'])
            ['transaction_amount_usd'].mean()
            .reset_index(name='hist_mean')
        )
        merged = latest.merge(
            hist_base,
            on=['customer_id', 'channel_group', 'debit_credit_indicator'],
            how='left'
        )
        merged['pct_change'] = (
            (merged['transaction_amount_usd'] - merged['hist_mean'])
            / (merged['hist_mean'] + 1e-6)
        )
        return merged
 
    pct_df = _pct(monthly)
 
    # International 
    intl_monthly = (
        df[df['is_international']]
        .groupby(['customer_id', 'txn_month', 'debit_credit_indicator'], as_index=False)
        ['transaction_amount_usd'].sum()
        .assign(channel_group='international')
    )
    intl_pct = _pct(intl_monthly)
 
    all_pct = pd.concat([pct_df, intl_pct], ignore_index=True)
 
    features = (
        all_pct
        .assign(feat=lambda x:
            x['channel_group'] + '_' +
            x['debit_credit_indicator'].str.lower() +
            '_pct_change'
        )
        .pivot_table(
            index='customer_id', columns='feat',
            values='pct_change', aggfunc='first'
        )
        .reset_index()
    )
    features.columns.name = None
    return features
 

In [ ]:
## Month-over-month ratios

def build_mom_ratios(
    df: pd.DataFrame,
    window_end: pd.Timestamp,
    lags: tuple = (1, 2, 3)
) -> pd.DataFrame:
    """
    For lags k ∈ {1, 2, 3}:
        ratio_kMoM = latest_month_amount / amount_k_months_ago
    Per (channel_group × debit_credit_indicator) and for international.

    Feature naming: {channel}_{direction}_ratio{k}MoM
    """
    monthly = (
        df.groupby(
            ['customer_id', 'txn_month', 'channel_group', 'debit_credit_indicator'],
            as_index=False
        )['transaction_amount_usd'].sum()
        .sort_values(['customer_id', 'channel_group', 'debit_credit_indicator', 'txn_month'])
    )
    latest_month = monthly['txn_month'].max()

    def _ratios(sub: pd.DataFrame) -> pd.DataFrame:
        for k in lags:
            sub = sub.copy()
            sub[f'ratio_{k}MoM'] = (
                sub['transaction_amount_usd']
                / sub.groupby(['customer_id', 'channel_group', 'debit_credit_indicator'])
                     ['transaction_amount_usd'].shift(k)
            )
        return sub[sub['txn_month'] == latest_month]

    ratio_df = _ratios(monthly)

    intl_monthly = (
        df[df['is_international']]
        .groupby(['customer_id', 'txn_month', 'debit_credit_indicator'], as_index=False)
        ['transaction_amount_usd'].sum()
        .assign(channel_group='international')
        .sort_values(['customer_id', 'debit_credit_indicator', 'txn_month'])
    )
    intl_ratio = _ratios(intl_monthly)

    all_ratio = pd.concat([ratio_df, intl_ratio], ignore_index=True)
    all_ratio['base_feat'] = (
        all_ratio['channel_group'] + '_' +
        all_ratio['debit_credit_indicator'].str.lower()
    )

    frames = {}
    for k in lags:
        frames[k] = (
            all_ratio
            .pivot_table(
                index='customer_id', columns='base_feat',
                values=f'ratio_{k}MoM', aggfunc='first'
            )
            .add_suffix(f'_ratio{k}MoM')
        )

    features = (
        pd.concat(frames.values(), axis=1)
        .reset_index()
        .replace([np.inf, -np.inf], np.nan)
    )
    features.columns.name = None
    return features


In [ ]:
## Ratio shift 
def build_ratio_shift(df: pd.DataFrame, window_end: pd.Timestamp) -> pd.DataFrame:
    """
    Two shift types, both measured as (latest_month_value − prior_month_value):
      • credit_debit_ratio_shift    : shift in the credit/debit amount ratio
      • {channel}_usage_ratio_shift : shift in each channel's share of total spend
      • international_usage_ratio_shift

    Feature naming: credit_debit_ratio_shift
                    {channel}_usage_ratio_shift
    """
    # ── Credit / Debit ratio shift ──────────────────────────────────────────
    monthly_totals = (
        df.groupby(['customer_id', 'txn_month', 'debit_credit_indicator'], as_index=False)
        ['transaction_amount_usd'].sum()
        .pivot_table(
            index=['customer_id', 'txn_month'],
            columns='debit_credit_indicator',
            values='transaction_amount_usd',
            fill_value=0
        )
        .reset_index()
    )
    monthly_totals['credit_debit_ratio'] = (
        monthly_totals.get('Credit', 0)
        / (monthly_totals.get('Debit', 0) + 1e-6)
    )

    latest_month = monthly_totals['txn_month'].max()
    prior_month  = latest_month - 1

    cd_pivot = (
        monthly_totals[monthly_totals['txn_month'].isin([latest_month, prior_month])]
        .pivot(index='customer_id', columns='txn_month', values='credit_debit_ratio')
    )
    cd_shift = pd.DataFrame({'customer_id': cd_pivot.index})
    cd_shift['credit_debit_ratio_shift'] = (
        cd_pivot.get(latest_month, np.nan) - cd_pivot.get(prior_month, np.nan)
    )
    cd_shift = cd_shift.reset_index(drop=True)

    # ── Channel usage ratio shift ────────────────────────────────────────────
    channel_monthly = (
        df.groupby(['customer_id', 'txn_month', 'channel_group'], as_index=False)
        ['transaction_amount_usd'].sum()
    )
    total_monthly = (
        channel_monthly.groupby(['customer_id', 'txn_month'], as_index=False)
        ['transaction_amount_usd'].sum()
        .rename(columns={'transaction_amount_usd': 'total_amt'})
    )
    channel_monthly = channel_monthly.merge(total_monthly, on=['customer_id', 'txn_month'])
    channel_monthly['usage_ratio'] = (
        channel_monthly['transaction_amount_usd'] / (channel_monthly['total_amt'] + 1e-6)
    )

    # International usage ratio
    intl_monthly = (
        df[df['is_international']]
        .groupby(['customer_id', 'txn_month'], as_index=False)
        ['transaction_amount_usd'].sum()
        .merge(total_monthly, on=['customer_id', 'txn_month'])
    )
    intl_monthly['usage_ratio']   = (
        intl_monthly['transaction_amount_usd'] / (intl_monthly['total_amt'] + 1e-6)
    )
    intl_monthly['channel_group'] = 'international'

    channel_usage = pd.concat(
        [
            channel_monthly[['customer_id', 'txn_month', 'channel_group', 'usage_ratio']],
            intl_monthly[['customer_id', 'txn_month', 'channel_group', 'usage_ratio']],
        ],
        ignore_index=True
    )

    usage_shift = (
        channel_usage[channel_usage['txn_month'].isin([latest_month, prior_month])]
        .pivot_table(
            index='customer_id',
            columns=['channel_group', 'txn_month'],
            values='usage_ratio'
        )
    )

    shift_records = {'customer_id': usage_shift.index}
    for ch in usage_shift.columns.get_level_values(0).unique():
        if (ch, latest_month) in usage_shift.columns and (ch, prior_month) in usage_shift.columns:
            shift_records[f'{ch}_usage_ratio_shift'] = (
                usage_shift[(ch, latest_month)] - usage_shift[(ch, prior_month)]
            )

    usage_shift_features = pd.DataFrame(shift_records).reset_index(drop=True)
    features = cd_shift.merge(usage_shift_features, on='customer_id', how='outer')
    return features



In [ ]:
## Volatility
def build_volatility(df: pd.DataFrame, window_end: pd.Timestamp) -> pd.DataFrame:
    """
    CV = std(monthly_amounts) / (mean(monthly_amounts) + ε)
    Computed per (channel_group × debit_credit_indicator) over all months in the
    window, and for international × direction.

    Feature naming: {channel}_{direction}_volatility_cv
    """
    monthly = (
        df.groupby(
            ['customer_id', 'txn_month', 'channel_group', 'debit_credit_indicator'],
            as_index=False
        )['transaction_amount_usd'].sum()
    )

    def cv(x):
        return x.std() / (x.mean() + 1e-6)

    cv_specs = [
        ('cash',    'Credit'), ('cash',    'Debit'),
        ('wire',    'Credit'), ('wire',    'Debit'),
        ('card',    'Debit'),
        ('atm',     'Debit'),
        ('digital', 'Credit'), ('digital', 'Debit'),
        ('ach',     'Credit'), ('ach',     'Debit'),
    ]

    frames = []
    for ch, cdi in cv_specs:
        feat_name = f'{ch}_{cdi.lower()}_volatility_cv'
        temp = (
            monthly[
                (monthly['channel_group'] == ch) &
                (monthly['debit_credit_indicator'] == cdi)
            ]
            .groupby('customer_id')['transaction_amount_usd']
            .apply(cv)
            .reset_index(name=feat_name)
        )
        frames.append(temp)

    # International
    intl_monthly = (
        df[df['is_international']]
        .groupby(['customer_id', 'txn_month', 'debit_credit_indicator'], as_index=False)
        ['transaction_amount_usd'].sum()
    )
    for cdi in ('Credit', 'Debit'):
        feat_name = f'international_{cdi.lower()}_volatility_cv'
        temp = (
            intl_monthly[intl_monthly['debit_credit_indicator'] == cdi]
            .groupby('customer_id')['transaction_amount_usd']
            .apply(cv)
            .reset_index(name=feat_name)
        )
        frames.append(temp)

    features = reduce(lambda l, r: l.merge(r, on='customer_id', how='outer'), frames)
    return features


In [ ]:
## Velocity Spike (count)

def build_velocity_spike_count(df: pd.DataFrame, window_end: pd.Timestamp) -> pd.DataFrame:
    """
    spike = (latest_month_count − avg_prior_2_months_count)
            / (avg_prior_2_months_count + ε)
    Per channel_group and for international.

    Feature naming: {channel}_txn_velocity_spike
                    international_txn_velocity_spike
    """
    monthly_cnt = (
        df.groupby(['customer_id', 'txn_month', 'channel_group'])
        .size()
        .reset_index(name='txn_count')
    )
    latest_month = monthly_cnt['txn_month'].max()
    prior1, prior2 = latest_month - 1, latest_month - 2

    def _spike(sub: pd.DataFrame, feat_name: str) -> pd.DataFrame:
        pivot = (
            sub.pivot_table(
                index='customer_id', columns='txn_month',
                values='txn_count', aggfunc='sum'
            ).fillna(0)
        )
        latest = pivot.get(
                            latest_month,
                            pd.Series(0, index=pivot.index)
                        )
        prior1_vals = pivot.get(
                                    prior1,
                                    pd.Series(0, index=pivot.index)
                                )
        prior2_vals = pivot.get(
                                    prior2,
                                    pd.Series(0, index=pivot.index)
                                )
        prior_avg = (prior1_vals + prior2_vals) / 2
        spike = ((latest - prior_avg) / (prior_avg + 1e-6))

        return spike.reset_index(name=feat_name)

    frames = []
    for ch in CHANNEL_MAP:
        frames.append(
            _spike(
                monthly_cnt[monthly_cnt['channel_group'] == ch],
                f'{ch}_txn_velocity_spike'
            )
        )

    # International
    intl_cnt = (
        df[df['is_international']]
        .groupby(['customer_id', 'txn_month']).size()
        .reset_index(name='txn_count')
    )
    pivot_i    = intl_cnt.pivot_table(
        index='customer_id', columns='txn_month',
        values='txn_count', aggfunc='sum'
    ).fillna(0)
    latest_i   = pivot_i.get(latest_month, 0)
    prior_avg_i = (pivot_i.get(prior1, 0) + pivot_i.get(prior2, 0)) / 2
    frames.append(
        ((latest_i - prior_avg_i) / (prior_avg_i + 1e-6))
        .reset_index(name='international_txn_velocity_spike')
    )

    base = df[['customer_id']].drop_duplicates()
    for f in frames:
        base = base.merge(f, on='customer_id', how='left')
    return base.fillna(0)


In [ ]:
## Amount velocity spike
def build_velocity_spike_amount(df: pd.DataFrame, window_end: pd.Timestamp) -> pd.DataFrame:
    """
    Computed per (channel_group × debit_credit_indicator) and international.

    Feature naming: {channel}_{direction}_amt_velocity_spike
    """
    monthly_amt = (
        df.groupby(
            ['customer_id', 'txn_month', 'channel_group', 'debit_credit_indicator'],
            as_index=False
        )['transaction_amount_usd'].sum()
    )
    latest_month = monthly_amt['txn_month'].max()
    prior1, prior2 = latest_month - 1, latest_month - 2

    def _spike(sub: pd.DataFrame, feat_name: str) -> pd.DataFrame:
        pivot = (
            sub.pivot_table(
                index='customer_id', columns='txn_month',
                values='transaction_amount_usd', aggfunc='sum'
            ).fillna(0)
        )
        latest = pivot.get(
                            latest_month,
                            pd.Series(0, index=pivot.index)
                        )
        prior1_vals = pivot.get(
                                    prior1,
                                    pd.Series(0, index=pivot.index)
                                )
        prior2_vals = pivot.get(
                                    prior2,
                                    pd.Series(0, index=pivot.index)
                                )
        prior_avg = (prior1_vals + prior2_vals) / 2
        spike = ((latest - prior_avg) / (prior_avg + 1e-6))

        return spike.reset_index(name=feat_name)

    specs = [
        ('cash',    'Credit'), ('cash',    'Debit'),
        ('wire',    'Credit'), ('wire',    'Debit'),
        ('card',    'Debit'),
        ('atm',     'Debit'),
        ('digital', 'Credit'), ('digital', 'Debit'),
        ('ach',     'Credit'), ('ach',     'Debit'),
    ]

    frames = []
    for ch, cdi in specs:
        sub = monthly_amt[
            (monthly_amt['channel_group'] == ch) &
            (monthly_amt['debit_credit_indicator'] == cdi)
        ]
        frames.append(_spike(sub, f'{ch}_{cdi.lower()}_amt_velocity_spike'))

    # International
    intl_monthly = (
        df[df['is_international']]
        .groupby(['customer_id', 'txn_month', 'debit_credit_indicator'], as_index=False)
        ['transaction_amount_usd'].sum()
    )
    for cdi in ('Credit', 'Debit'):
        sub = intl_monthly[intl_monthly['debit_credit_indicator'] == cdi].copy()
        sub['channel_group'] = 'international'         
        frames.append(_spike(sub, f'international_{cdi.lower()}_amt_velocity_spike'))

    base = df[['customer_id']].drop_duplicates()
    for f in frames:
        base = base.merge(f, on='customer_id', how='left')
    return base.fillna(0)


In [ ]:
## Trend deviation
def build_trend_deviation(df: pd.DataFrame, window_end: pd.Timestamp) -> pd.DataFrame:
    """
    Fit a linear (OLS) trend over the 3 months *prior* to the latest month in
    the window; then measure:
        trend_deviation = (actual_latest − expected_latest) / (expected + ε)

    Computed for: total Credit, total Debit, cash, wire, digital.

    Feature naming: total_credit_trend_deviation, total_debit_trend_deviation,
                    cash_amount_trend_deviation, wire_amount_trend_deviation,
                    digital_amount_trend_deviation
    """
    latest_month = df['txn_month'].max()
    trend_months = [latest_month - 3, latest_month - 2, latest_month - 1]

    monthly_amt = (
        df.groupby(
            ['customer_id', 'txn_month', 'channel_group', 'debit_credit_indicator'],
            as_index=False
        )['transaction_amount_usd'].sum()
    )

    def _deviation(pivot_df: pd.DataFrame, feat_name: str) -> pd.DataFrame:
        deviations = []
        for party, row in pivot_df.iterrows():
            y = row.values.astype(float)
            x = np.arange(len(y))
            if np.count_nonzero(y) < 2:
                deviations.append(0.0)
                continue
            slope, intercept = np.polyfit(x, y, 1)
            expected = slope * len(y) + intercept     
            actual   = y[-1]
            deviations.append((actual - expected) / (abs(expected) + 1e-6))
        return pd.DataFrame({'customer_id': pivot_df.index, feat_name: deviations})

    # Specs: (channel_group filter, debit_credit_indicator filter, feature_name)
    trend_specs = [
        (None,      'Credit', 'total_credit_trend_deviation'),
        (None,      'Debit',  'total_debit_trend_deviation'),
        ('cash',    None,     'cash_amount_trend_deviation'),
        ('wire',    None,     'wire_amount_trend_deviation'),
        ('digital', None,     'digital_amount_trend_deviation'),
    ]

    frames = []
    for ch, cdi, feat_name in trend_specs:
        mask = pd.Series([True] * len(monthly_amt), index=monthly_amt.index)
        if ch  is not None: mask &= (monthly_amt['channel_group'] == ch)
        if cdi is not None: mask &= (monthly_amt['debit_credit_indicator'] == cdi)

        pivot = (
            monthly_amt[mask]
            .groupby(['customer_id', 'txn_month'])['transaction_amount_usd']
            .sum()
            .unstack()
            .reindex(columns=trend_months, fill_value=0)
        )
        frames.append(_deviation(pivot, feat_name))

    base = df[['customer_id']].drop_duplicates()
    for f in frames:
        base = base.merge(f, on='customer_id', how='left')
    return base.fillna(0)


In [ ]:
## Segment comparison
def build_segment_comparison(
    df: pd.DataFrame,
    window_end: pd.Timestamp,
    cluster_label: int
) -> pd.DataFrame:
    """
    Because `df` is already filtered to ONE cluster, segment statistics (median /
    mean) are computed from the slice itself — no cross-cluster leakage.

    Ratios:
      amount_vs_seg_ratio = customer_avg_amount / cluster_median_avg_amount
      volume_vs_seg_ratio = customer_volume     / cluster_mean_volume

    Per channel_group and for international.

    Feature naming: {channel}_amount_vs_seg_ratio, {channel}_volume_vs_seg_ratio
                    intl_amount_vs_seg_ratio,       intl_volume_vs_seg_ratio
    """
    agg = (
        df.groupby(['customer_id', 'channel_group'])
        .agg(
            amount_mean=('transaction_amount_usd', 'mean'),
            volume=('transaction_amount_usd', 'count')
        )
        .reset_index()
    )

    seg_median = agg.groupby('channel_group')[['amount_mean', 'volume']].median().reset_index()
    seg_mean   = agg.groupby('channel_group')[['amount_mean', 'volume']].mean().reset_index()

    agg = (
        agg
        .merge(seg_median, on='channel_group', suffixes=('', '_seg_med'))
        .merge(seg_mean,   on='channel_group', suffixes=('', '_seg_mean'))
    )
    agg['amount_vs_seg_ratio'] = agg['amount_mean'] / (agg['amount_mean_seg_med']  + 1e-6)
    agg['volume_vs_seg_ratio'] = agg['volume']       / (agg['volume_seg_mean']      + 1e-6)

    amount_pivot = (
        agg.pivot_table(
            index='customer_id', columns='channel_group',
            values='amount_vs_seg_ratio', aggfunc='first'
        )
        .add_suffix('_amount_vs_seg_ratio')
    )
    volume_pivot = (
        agg.pivot_table(
            index='customer_id', columns='channel_group',
            values='volume_vs_seg_ratio', aggfunc='first'
        )
        .add_suffix('_volume_vs_seg_ratio')
    )

    # International
    intl_agg = (
        df[df['is_international']]
        .groupby('customer_id')
        .agg(
            intl_amount_mean=('transaction_amount_usd', 'mean'),
            intl_volume=('transaction_amount_usd', 'count')
        )
        .reset_index()
    )
    seg_intl_med_amt = intl_agg['intl_amount_mean'].median()
    seg_intl_mean_vol = intl_agg['intl_volume'].mean()
    intl_agg['intl_amount_vs_seg_ratio'] = (
        intl_agg['intl_amount_mean'] / (seg_intl_med_amt   + 1e-6)
    )
    intl_agg['intl_volume_vs_seg_ratio'] = (
        intl_agg['intl_volume']       / (seg_intl_mean_vol  + 1e-6)
    )
    intl_feat = intl_agg[['customer_id', 'intl_amount_vs_seg_ratio', 'intl_volume_vs_seg_ratio']]

    features = (
        pd.concat([amount_pivot, volume_pivot], axis=1)
        .reset_index()
        .merge(intl_feat, on='customer_id', how='left')
    )
    features.columns.name = None
    return features


In [ ]:
## New geography

def build_new_geography(df: pd.DataFrame, window_end: pd.Timestamp) -> pd.DataFrame:
    """
    Detects countries in the *latest month* that the customer had never
    transacted with in prior months of the window.

    Expects columns: counterparty_country (str), is_high_risk_country (bool).

    Feature naming: new_country_txn_count, new_high_risk_country_flag
    """
    latest_month = df['txn_month'].max()
    df_latest    = df[df['txn_month'] == latest_month].copy()
    df_prior     = df[df['txn_month'] <  latest_month].copy()

    prior_countries = (
        df_prior.groupby('customer_id')['counterparty_country']
        .apply(set).to_dict()
    )

    def _geo(group):
        seen   = prior_countries.get(group.name, set())
        is_new = ~group['counterparty_country'].isin(seen)
        is_new_hr = is_new & group['is_high_risk_country']
        return pd.Series({
            'new_country_txn_count':      int(is_new.sum()),
            'new_high_risk_country_flag': int(is_new_hr.any()),
        })

    features = df_latest.groupby('customer_id').apply(_geo).reset_index()
    return features


In [ ]:
## Timing anomaly
def build_timing_anomaly(df: pd.DataFrame, window_end: pd.Timestamp) -> pd.DataFrame:
    """
    Share of transactions that occur at unusual times:
      odd_hour_txn_ratio  : fraction before 08:00 or after 20:00
      weekend_txn_ratio   : fraction on Saturday / Sunday
      holiday_txn_flag    : 1 if any transaction fell on a configured holiday

    Feature naming: odd_hour_txn_ratio, weekend_txn_ratio, holiday_txn_flag
    """
    d = df.copy()
    d['hour']        = d['transaction_datetime'].dt.hour
    d['dow']         = d['transaction_datetime'].dt.dayofweek   # Mon=0, Sun=6
    d['is_weekend']  = d['dow'] >= 5
    d['is_odd_hour'] = (d['hour'] < 8) | (d['hour'] >= 20)
    # d['is_holiday']  = d['transaction_datetime'].dt.normalize().isin(HOLIDAYS) ## Holiday list needs to be updated

    features = (
        d.groupby('customer_id')
        .apply(lambda x: pd.Series({
            'odd_hour_txn_ratio': x['is_odd_hour'].mean(),
            'weekend_txn_ratio':  x['is_weekend'].mean()
            # 'holiday_txn_flag':   int(x['is_holiday'].any()),
        }))
        .reset_index()
    )
    return features


In [ ]:
## New counter party
def build_new_counterparty(df: pd.DataFrame, window_end: pd.Timestamp) -> pd.DataFrame:
    """
    Compares latest month's counterparty set against prior months.

    Feature naming:
      new_counterparty_txn_count    : # transactions to first-seen counterparties
      new_counterparty_amount_ratio : total amount to new cps / total amount
      single_cp_dominance_pct       : top-1 cp amount / total amount (latest month)

    Expects column: counterparty_account (str)
    """
    latest_month = df['txn_month'].max()
    df_latest    = df[df['txn_month'] == latest_month].copy()
    df_prior     = df[df['txn_month'] <  latest_month].copy()

    prior_cps = (
        df_prior.groupby('customer_id')['counterparty_account']
        .apply(set).to_dict()
    )

    def _cp(group):
        seen       = prior_cps.get(group.name, set())
        group      = group.copy()
        group['is_new'] = ~group['counterparty_account'].isin(seen)
        total_amt  = group['transaction_amount_usd'].sum()
        new_amt    = group.loc[group['is_new'], 'transaction_amount_usd'].sum()
        cp_amt     = group.groupby('counterparty_account')['transaction_amount_usd'].sum()
        dominance  = cp_amt.max() / (total_amt + 1e-6)
        return pd.Series({
            'new_counterparty_txn_count':    int(group['is_new'].sum()),
            'new_counterparty_amount_ratio': new_amt / (total_amt + 1e-6),
            'single_cp_dominance_pct':       dominance,
        })

    features = df_latest.groupby('customer_id').apply(_cp).reset_index()
    return features



In [ ]:
## Counterparty behaviour
def build_counterparty_behaviour(df: pd.DataFrame, window_end: pd.Timestamp) -> pd.DataFrame:
    """
    Measures counterparty concentration and velocity over the full window.

    Feature naming:
      counterparty_concentration_ratio      : top-1 cp amount share
      counterparty_concentration_ratio_top5 : top-5 cp amount share
      counterparty_velocity_spike           : spike in max single-cp txn count
                                             (latest vs avg of 2 prior months)
    """
    latest_month   = df['txn_month'].max()
    prior1, prior2 = latest_month - 1, latest_month - 2

    cp_amt = (
        df.groupby(['customer_id', 'counterparty_account'])
        ['transaction_amount_usd'].sum().reset_index()
    )
    total_amt = (
        cp_amt.groupby('customer_id')['transaction_amount_usd']
        .sum().reset_index(name='total_amt')
    )
    top1 = (
        cp_amt.groupby('customer_id')['transaction_amount_usd']
        .max().reset_index(name='top1_amt')
    )
    top5 = (
        cp_amt
        .sort_values(['customer_id', 'transaction_amount_usd'], ascending=[True, False])
        .groupby('customer_id').head(5)
        .groupby('customer_id')['transaction_amount_usd'].sum()
        .reset_index(name='top5_amt')
    )

    conc = total_amt.merge(top1, on='customer_id').merge(top5, on='customer_id')
    conc['counterparty_concentration_ratio']      = conc['top1_amt'] / (conc['total_amt'] + 1e-6)
    conc['counterparty_concentration_ratio_top5'] = conc['top5_amt'] / (conc['total_amt'] + 1e-6)
    conc_features = conc[['customer_id',
                           'counterparty_concentration_ratio',
                           'counterparty_concentration_ratio_top5']]

    # Velocity spike on max single-cp txn count
    monthly_cp_cnt = (
        df.groupby(['customer_id', 'txn_month', 'counterparty_account'])
        .size().reset_index(name='txn_count')
    )
    monthly_max_cp = (
        monthly_cp_cnt.groupby(['customer_id', 'txn_month'])['txn_count']
        .max().reset_index(name='max_cp_count')
    )
    latest_vals = (
        monthly_max_cp[monthly_max_cp['txn_month'] == latest_month]
        .rename(columns={'max_cp_count': 'latest_count'})
    )
    prior_avg = (
        monthly_max_cp[monthly_max_cp['txn_month'].isin([prior1, prior2])]
        .groupby('customer_id')['max_cp_count'].mean()
        .reset_index(name='prior_avg_count')
    )
    vel = latest_vals.merge(prior_avg, on='customer_id', how='left')
    vel['counterparty_velocity_spike'] = (
        (vel['latest_count'] - vel['prior_avg_count'])
        / (vel['prior_avg_count'] + 1e-6)
    )
    vel_features = vel[['customer_id', 'counterparty_velocity_spike']]

    features = conc_features.merge(vel_features, on='customer_id', how='left')
    return features


In [ ]:
## Pattern ratio 
def build_pattern_ratio(df: pd.DataFrame, window_end: pd.Timestamp) -> pd.DataFrame:
    """
    Two structuring-behaviour indicators:
      round_amount_ratio_shift : change in fraction of round-amount (÷100) txns
                                 (latest month vs avg of 2 prior months)
      repeated_amount_ratio    : fraction of txns where the exact amount
                                 appears more than once for that customer

    Requires is_round_amount column (added in prep_window_df).
    """
    latest_month   = df['txn_month'].max()
    prior1, prior2 = latest_month - 1, latest_month - 2

    monthly_round = (
        df.groupby(['customer_id', 'txn_month'])
        .agg(round_count=('is_round_amount', 'sum'),
             total_count=('is_round_amount', 'count'))
        .reset_index()
    )
    monthly_round['round_ratio'] = (
        monthly_round['round_count'] / (monthly_round['total_count'] + 1e-6)
    )

    latest_rr = (
        monthly_round[monthly_round['txn_month'] == latest_month]
        .set_index('customer_id')['round_ratio'].rename('rr_latest')
    )
    prior_rr = (
        monthly_round[monthly_round['txn_month'].isin([prior1, prior2])]
        .groupby('customer_id')['round_ratio'].mean().rename('rr_prior_avg')
    )
    rr_shift = pd.concat([latest_rr, prior_rr], axis=1).reset_index()
    rr_shift['round_amount_ratio_shift'] = (
        (rr_shift['rr_latest'] - rr_shift['rr_prior_avg'])
        / (rr_shift['rr_prior_avg'] + 1e-6)
    )
    rr_shift = rr_shift[['customer_id', 'round_amount_ratio_shift']]

    # Repeated exact amounts
    amt_freq = (
        df.groupby(['customer_id', 'transaction_amount_usd'])
        .size().reset_index(name='amt_count')
    )
    repeated = amt_freq[amt_freq['amt_count'] > 1]
    rep_sum  = (
        repeated.groupby('customer_id')['amt_count']
        .sum().rename('repeated_txn_count').to_frame().reset_index()
    )
    total_txns = (
        df.groupby('customer_id').size().rename('total_txn_count')
        .to_frame().reset_index()
    )
    rep_ratio = total_txns.merge(rep_sum, on='customer_id', how='left').fillna(0)
    rep_ratio['repeated_amount_ratio'] = (
        rep_ratio['repeated_txn_count'] / (rep_ratio['total_txn_count'] + 1e-6)
    )
    rep_ratio = rep_ratio[['customer_id', 'repeated_amount_ratio']]

    features = rr_shift.merge(rep_ratio, on='customer_id', how='left')
    return features


In [ ]:
## Activity gap 
def build_activity_gap(df: pd.DataFrame, window_end: pd.Timestamp) -> pd.DataFrame:
    """
    inactivity_gap_days       : maximum gap (in days) between consecutive txns
    post_inactivity_spike_flag: 1 if a burst ≥ avg_daily_rate × 7 days occurs
                                within 7 days after a ≥ 30-day gap

    Feature naming: inactivity_gap_days, post_inactivity_spike_flag
    """
    inactive_days     = 30
    spike_window_days = 7

    d = df.sort_values(['customer_id', 'transaction_datetime']).copy()
    d['txn_gap_days'] = (
        d.groupby('customer_id')['transaction_datetime'].diff().dt.days
    )

    inactivity_gap = (
        d.groupby('customer_id')['txn_gap_days']
        .max().fillna(0).reset_index(name='inactivity_gap_days')
    )

    avg_daily_rate = (
        d.groupby('customer_id').apply(lambda g: (
            len(g)
            / max((g['transaction_datetime'].max()
                   - g['transaction_datetime'].min()).days, 1)
        ))
    )

    inactivity_pts = d[d['txn_gap_days'] >= inactive_days].copy()

    post_gap = d.merge(
        inactivity_pts[['customer_id', 'transaction_datetime']],
        on='customer_id',
        suffixes=('', '_gap')
    )
    post_gap = post_gap[
        (post_gap['transaction_datetime'] > post_gap['transaction_datetime_gap'])
        & (post_gap['transaction_datetime'] <=
           post_gap['transaction_datetime_gap'] + pd.Timedelta(days=spike_window_days))
    ]
    post_gap_counts = post_gap.groupby('customer_id').size().rename('post_gap_txn_count')

    spike_df = (
        avg_daily_rate.to_frame('avg_daily_rate')
        .merge(post_gap_counts, left_index=True, right_index=True, how='left')
        .fillna(0)
        .reset_index()
        .rename(columns={'index': 'customer_id'})
    )
    spike_df['post_inactivity_spike_flag'] = (
        spike_df['post_gap_txn_count']
        >= spike_df['avg_daily_rate'] * spike_window_days
    ).astype(int)
    spike_df = spike_df[['customer_id', 'post_inactivity_spike_flag']]

    features = inactivity_gap.merge(spike_df, on='customer_id', how='left')
    return features


In [ ]:
FEATURE_BUILDERS = [
    ('pct_change',             build_pct_change),
    ('mom_ratios',             build_mom_ratios),
    ('ratio_shift',            build_ratio_shift),
    ('volatility',             build_volatility),
    ('velocity_spike_count',   build_velocity_spike_count),
    ('velocity_spike_amount',  build_velocity_spike_amount),
    ('trend_deviation',        build_trend_deviation),
    ('new_geography',          build_new_geography),
    ('timing_anomaly',         build_timing_anomaly),
    ('new_counterparty',       build_new_counterparty),
    ('counterparty_behaviour', build_counterparty_behaviour),
    ('pattern_ratio',          build_pattern_ratio),
    ('activity_gap',           build_activity_gap),
]


In [ ]:
def build_anomaly_features_for_window_cluster(
    df_window_cluster: pd.DataFrame,
    window_end: pd.Timestamp,
    cluster_label: int
) -> pd.DataFrame:
    """
    Run all 14 feature groups on a single (window × cluster) slice.

    Parameters
    ----------
    df_window_cluster : prepped window data filtered to ONE cluster_label.
    window_end        : the window's end date.
    cluster_label     : integer cluster id (passed to build_segment_comparison).

    Returns
    -------
    customer-level DataFrame — one row per customer_id with all feature columns.
    """
    base = df_window_cluster[['customer_id']].drop_duplicates()

    # Run the 13 generic feature groups
    for name, builder in FEATURE_BUILDERS:
        try:
            feat = builder(df_window_cluster, window_end)
            base = base.merge(feat, on='customer_id', how='left')
        except Exception as exc:
            print(f"    [WARN] Feature group '{name}' failed for cluster {cluster_label}: {exc}")

    # Segment comparison — requires cluster context
    try:
        seg_feat = build_segment_comparison(df_window_cluster, window_end, cluster_label)
        base     = base.merge(seg_feat, on='customer_id', how='left')
    except Exception as exc:
        print(f"    [WARN] Segment comparison failed for cluster {cluster_label}: {exc}")

    return base



In [ ]:
def aggregate_across_windows(window_feature_dfs: list) -> pd.DataFrame:
    """
    Concatenate per-window feature tables and produce one row per customer
    by averaging numeric features across windows.

    Also appends:
      window_count : number of windows in which the customer appeared.

    Parameters
    ----------
    window_feature_dfs : list of DataFrames, one per window.
                         Each must have [customer_id, cluster_label, <features>].

    Returns
    -------
    Single customer-level DataFrame.
    """
    all_windows = pd.concat(window_feature_dfs, ignore_index=True)

    meta_cols    = ['customer_id', 'cluster_label']
    numeric_cols = [c for c in all_windows.columns if c not in meta_cols]

    # Mean across windows for all numeric features
    agg_features = (
        all_windows.groupby('customer_id')[numeric_cols]
        .mean()
        .reset_index()
    )

    # Record how many windows each customer appeared in
    window_count = (
        all_windows.groupby('customer_id').size()
        .reset_index(name='window_count')
    )
    agg_features = agg_features.merge(window_count, on='customer_id', how='left')

    # Re-attach cluster_label (most frequent value across windows per customer)
    cluster_mode = (
        all_windows.groupby('customer_id')['cluster_label']
        .agg(lambda x: x.mode().iloc[0] if not x.mode().empty else np.nan)
        .reset_index()
    )
    agg_features = agg_features.merge(cluster_mode, on='customer_id', how='left')

    return agg_features


In [ ]:
print(f"\ndataset_type : {dataset_type}")
print(f"date_folder  : {date_folder_fmt}")
print(f"DATA_PATH    : {DATA_PATH}")

# TRAIN path
# Each of the 3 windows is read, split by cluster, features are built per
# (window × cluster), concatenated across clusters for that window, and
# finally averaged across all windows.
if dataset_type == 'train':

    df_cluster = read_parquet_from_azure(
        container,
        f"{DATA_PATH}/cluster_description_mapping_train.parquet"
    )
    # Expected columns: customer_id, cluster_label, cluster_desc

    window_feature_dfs = []   # collects one aggregated DF per window

    for w in windows_info:
        window_name  = w['window']
        window_start = pd.Timestamp(w['start_date'])
        window_end   = pd.Timestamp(w['end_date'])

        print(f"\n=== Window: {window_name}  [{window_start.date()} → {window_end.date()}] ===")

        df_window_raw = read_parquet_from_azure(
            container,
            f"{DATA_PATH}/tms_data_{window_name}.parquet"
        )
        df_window = prep_window_df(df_window_raw)

        # Attach cluster labels for this window
        df_window = df_window.merge(
            df_cluster[['customer_id', 'cluster_label', 'cluster_desc']],
            on='customer_id', how='left'
        )

        cluster_feature_dfs = []   # one DF per cluster for this window

        for cl in CLUSTER_LABELS:
            print(f"  -- Cluster {cl} ...", end=' ', flush=True)
            df_cl = df_window[df_window['cluster_label'] == cl].copy()

            if df_cl.empty:
                print("empty — skipped")
                continue

            cl_features              = build_anomaly_features_for_window_cluster(df_cl, window_end, cl)
            cl_features['cluster_label'] = cl
            cluster_feature_dfs.append(cl_features)
            print(f"done  ({len(df_cl):,} rows → {len(cl_features):,} customers)")

        if cluster_feature_dfs:
            # Stack all clusters for this window into one DF
            window_all_clusters = pd.concat(cluster_feature_dfs, ignore_index=True)
            window_feature_dfs.append(window_all_clusters)

    # ── Average across windows ────────────────────────────────────────────────
    print("\n=== Aggregating across windows (mean per customer_id) ===")
    df_anomaly_features_train = aggregate_across_windows(window_feature_dfs)

    # Attach cluster_desc from the mapping file
    df_anomaly_features_train = df_anomaly_features_train.merge(
        df_cluster[['customer_id', 'cluster_desc']].drop_duplicates(),
        on='customer_id', how='left'
    )

    # Reorder: customer_id first, cluster cols last
    cluster_cols = ['cluster_label', 'cluster_desc']
    other_cols   = [c for c in df_anomaly_features_train.columns if c not in ['customer_id'] + cluster_cols]
    df_anomaly_features_train = df_anomaly_features_train[['customer_id'] + other_cols + cluster_cols]

    print(f"Final train feature shape : {df_anomaly_features_train.shape}")

    save_df_to_azure(
        df_anomaly_features_train,
        container,
        f"{DATA_PATH}/anomaly_features_train.parquet"
    )
    print("Saved → anomaly_features_train.parquet")
else:

    df_cluster = read_parquet_from_azure(
        container,
        f"{DATA_PATH}/cluster_description_mapping_test.parquet"
    )

    df_test_raw = read_parquet_from_azure(
        container,
        f"{DATA_PATH}/tms_data_test.parquet"
    )
    df_test = prep_window_df(df_test_raw)
    df_test = df_test.merge(
        df_cluster[['customer_id', 'cluster_label', 'cluster_desc']],
        on='customer_id', how='left'
    )

    test_window_end = df_test['transaction_datetime'].max()
    print(f"\n=== Test window_end: {test_window_end.date()} ===")

    cluster_feature_dfs = []

    for cl in CLUSTER_LABELS:
        print(f"  -- Test Cluster {cl} ...", end=' ', flush=True)
        df_cl = df_test[df_test['cluster_label'] == cl].copy()

        if df_cl.empty:
            print("empty — skipped")
            continue

        cl_features              = build_anomaly_features_for_window_cluster(df_cl, test_window_end, cl)
        cl_features['cluster_label'] = cl
        cluster_feature_dfs.append(cl_features)
        print(f"done  ({len(df_cl):,} rows → {len(cl_features):,} customers)")

    df_anomaly_features_test = pd.concat(cluster_feature_dfs, ignore_index=True)
    df_anomaly_features_test = df_anomaly_features_test.merge(
        df_cluster[['customer_id', 'cluster_desc']].drop_duplicates(),
        on='customer_id', how='left'
    )

    cluster_cols = ['cluster_label', 'cluster_desc']
    other_cols   = [c for c in df_anomaly_features_test.columns if c not in ['customer_id'] + cluster_cols]
    df_anomaly_features_test = df_anomaly_features_test[['customer_id'] + other_cols + cluster_cols]

    print(f"Final test feature shape : {df_anomaly_features_test.shape}")

    save_df_to_azure(
        df_anomaly_features_test,
        container,
        f"{DATA_PATH}/anomaly_features_test.parquet"
    )
    print("Saved → anomaly_features_test.parquet")

